# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SWAPI03/flyrank-ai-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule, its reason code, and the two signals it leans on

**The rule, in plain words.** A page is worth a title/meta review when it is *visible* (real
impressions), *well positioned* (already ranking on page one), yet earns *far less CTR than other
pages at its own position tier*. Rank those by how big the gap is, weighted by how much traffic is
at stake.

**Reason code (one):** `low_ctr_strong_position` — `impressions_90d >= 500`, `avg_position` in 1-20,
and `ctr` below half its position tier's median CTR.

**Action label:** `review_title_meta` (rewrite title / meta / snippet). Everything else stays off the
queue (implicit `monitor`).

**Score:** `gap * log1p(impressions_90d)`, where `gap = max(tier_median_ctr - ctr, 0)`. Transparent,
no fitted weights — big shortfalls on high-traffic pages rank first.

**Two signals this rule leans on (checked below, each flag-linked):**

1. **CTR vs position** (behind FlyRank's CTR-fix logic) — my rule only makes sense if position
   really drives CTR. Expected: CONFIRMED.
2. **Staleness vs CTR** (behind the refresh flags) — I test whether staleness *also* predicts low
   CTR, i.e. whether I should fold it into this rule. Expected: unclear, so let the data rule.

In [1]:
# Setup + load (starter data; no future-window or product-flag columns used).
import os, sys, json, subprocess
import pandas as pd, numpy as np
if "google.colab" in sys.modules and not os.path.isdir("data/raw"):
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
                        "flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

vis = df[(df["impressions_90d"] >= 100) & (df["ctr"].notna())].copy()
vis["tier_median_ctr"] = vis.groupby("position_tier")["ctr"].transform("median")

# ---- Signal 1: CTR by position tier (flag-linked: CTR-fix logic) ----
order = ["top_3", "striking", "page_1", "page_3_5", "deep"]
g = (vis.groupby("position_tier", observed=True)
        .agg(n=("ctr", "size"), mean_ctr=("ctr", "mean"), median_ctr=("ctr", "median")))
g = g.reindex([o for o in order if o in g.index])
print("SIGNAL 1 - CTR by position tier (n shown):")
print(g.round(4).to_string())
print("VERDICT: MIXED. CTR collapses at page_3_5 / deep, but the top tiers are noisy and not")
print("strictly ordered -> so my rule compares each page to its OWN tier median, never a global rank.\n")

# ---- Signal 2: staleness vs CTR (flag-linked: refresh flags) ----
vis["stale_bucket"] = pd.cut(vis["days_since_last_update"], [-1, 30, 90, 180, 365, 10**9],
                             labels=["0-30", "31-90", "91-180", "181-365", "365+"])
s = (vis.groupby("stale_bucket", observed=True)
        .agg(n=("ctr", "size"), mean_ctr=("ctr", "mean"), median_ctr=("ctr", "median")))
print("SIGNAL 2 - CTR by staleness bucket (n shown):")
print(s.round(4).to_string())
print("VERDICT: FALSE. No clean staleness->CTR relationship; the extreme buckets are tiny-n noise.")
print("A clearly-explained negative: staleness belongs to the refresh lane, so I keep it OUT of this")
print("CTR rule. That negative just saved the rule from a spurious input.")

SIGNAL 1 - CTR by position tier (n shown):
                  n  mean_ctr  median_ctr
position_tier                            
top_3           533    0.3341        0.19
striking       5903    0.2558        0.15
page_1         8633    0.3548        0.23
page_3_5       6058    0.1424        0.06
deep            879    0.0554        0.00
VERDICT: MIXED. CTR collapses at page_3_5 / deep, but the top tiers are noisy and not
strictly ordered -> so my rule compares each page to its OWN tier median, never a global rank.

SIGNAL 2 - CTR by staleness bucket (n shown):
                  n  mean_ctr  median_ctr
stale_bucket                             
0-30          13735    0.2712       0.150
31-90           152    0.1353       0.055
91-180         8084    0.2334       0.130
181-365          35    0.8406       0.180
VERDICT: FALSE. No clean staleness->CTR relationship; the extreme buckets are tiny-n noise.
A clearly-explained negative: staleness belongs to the refresh lane, so I keep it OUT of th

## 2. Build the ranked queue (writes the CSV)

Encode the one rule, attach the single reason code and action, rank by score, and write
`work/outputs/baseline_action_score.csv` (regenerated every run; kept out of git by the leak-guard).
I also write a small `baseline_metrics.json` receipt, which *is* worth committing.

In [2]:
# Encode the rule -> score, reason code, action, ranked queue.
vis["gap"] = (vis["tier_median_ctr"] - vis["ctr"]).clip(lower=0)
flag = ((vis["impressions_90d"] >= 500) & (vis["avg_position"] >= 1) &
        (vis["avg_position"] <= 20) & (vis["ctr"] < 0.5 * vis["tier_median_ctr"]))

queue = vis[flag].copy()
queue["score"] = queue["gap"] * np.log1p(queue["impressions_90d"])
queue["reason_code"] = "low_ctr_strong_position"
queue["action"] = "review_title_meta"
queue = queue.sort_values("score", ascending=False).reset_index(drop=True)

cols = ["content_id", "client_id", "content_type", "position_tier", "impressions_90d",
        "avg_position", "ctr", "tier_median_ctr", "gap", "score", "reason_code", "action"]
os.makedirs("work/outputs", exist_ok=True)
queue[cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

metrics = {
    "rule": "low_ctr_strong_position",
    "thresholds": {"min_impressions": 500, "position_min": 1, "position_max": 20,
                   "ctr_below_fraction_of_tier_median": 0.5},
    "visible_pool": int(len(vis)),
    "queue_size": int(len(queue)),
    "flagged_share_of_visible": round(len(queue) / len(vis), 3),
    "signal_verdicts": {"ctr_vs_position": "MIXED", "staleness_vs_ctr": "FALSE"},
}
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"queue written: {len(queue):,} pages -> work/outputs/baseline_action_score.csv")
print(f"metrics receipt -> work/outputs/baseline_metrics.json")
print(queue[["content_id", "impressions_90d", "avg_position", "position_tier",
             "ctr", "tier_median_ctr", "score", "action"]].head(10).round(3).to_string(index=False))

queue written: 2,920 pages -> work/outputs/baseline_action_score.csv
metrics receipt -> work/outputs/baseline_metrics.json
          content_id  impressions_90d  avg_position position_tier  ctr  tier_median_ctr  score            action
content_c8e9d6ab9013           208678           9.7        page_1 0.00             0.23  2.817 review_title_meta
content_453722754fea           140079           7.6        page_1 0.01             0.23  2.607 review_title_meta
content_39881853ef0c           112434           7.2        page_1 0.01             0.23  2.559 review_title_meta
content_c84a0ab98e90           223271           7.8        page_1 0.03             0.23  2.463 review_title_meta
content_0919dd345d80           119217           7.0        page_1 0.02             0.23  2.455 review_title_meta
content_d274ac4158ef            65138           6.8        page_1 0.01             0.23  2.439 review_title_meta
content_e5f459e737b7            56363           5.9        page_1 0.01             0.2

## 3. Top-20 review (action, reason, confidence, what would make it wrong)

Reading my own top of list with a skeptic's eye. For each row: the action, why it's there, a
confidence note, and the single most likely reason it could be a bad pick.

In [3]:
def what_would_make_it_wrong(r):
    if r["ctr"] == 0 and r["impressions_90d"] >= 5000:
        return "zero clicks at high impressions: possible tracking/attribution or brand-navigational mismatch, not a snippet fix"
    if r["content_type"] == "feedly article":
        return "feedly page lacks keyword/intent context; low CTR may be feed placement, not title/meta"
    if r["avg_position"] > 7 and r["position_tier"] == "page_1":
        return "bottom-of-page-1 rank; some CTR shortfall is expected for position, not a title problem"
    return "CTR gap may be a brand / SERP-feature / intent-mismatch effect rather than a fixable snippet"

review = queue.head(20).copy()
review["rank"] = review.index + 1
review["confidence"] = np.where(review["impressions_90d"] >= 3000, "high", "medium")
review["why"] = "ctr " + review["ctr"].round(2).astype(str) + " vs tier median " + review["tier_median_ctr"].round(2).astype(str)
review["what_would_make_it_wrong"] = review.apply(what_would_make_it_wrong, axis=1)

for _, r in review.iterrows():
    print(f"#{int(r['rank']):>2} {r['content_id']}  [{r['position_tier']}, pos {r['avg_position']:.1f}, imp {int(r['impressions_90d']):,}]")
    print(f"     action: {r['action']}  | reason: {r['reason_code']} ({r['why']})  | confidence: {r['confidence']}")
    print(f"     what would make it wrong: {r['what_would_make_it_wrong']}")

# 1 content_c8e9d6ab9013  [page_1, pos 9.7, imp 208,678]
     action: review_title_meta  | reason: low_ctr_strong_position (ctr 0.0 vs tier median 0.23)  | confidence: high
     what would make it wrong: zero clicks at high impressions: possible tracking/attribution or brand-navigational mismatch, not a snippet fix
# 2 content_453722754fea  [page_1, pos 7.6, imp 140,079]
     action: review_title_meta  | reason: low_ctr_strong_position (ctr 0.01 vs tier median 0.23)  | confidence: high
     what would make it wrong: bottom-of-page-1 rank; some CTR shortfall is expected for position, not a title problem
# 3 content_39881853ef0c  [page_1, pos 7.2, imp 112,434]
     action: review_title_meta  | reason: low_ctr_strong_position (ctr 0.01 vs tier median 0.23)  | confidence: high
     what would make it wrong: bottom-of-page-1 rank; some CTR shortfall is expected for position, not a title problem
# 4 content_c84a0ab98e90  [page_1, pos 7.8, imp 223,271]
     action: review_title_meta  | reason

## 4. Weak picks + leakage check

The top-20 review surfaces a systematic weak spot, and I confirm the rule uses no future-window or
label-derived inputs.

In [4]:
# Weak picks: the honest soft spot in this baseline.
weak = queue[(queue["avg_position"] > 7) & (queue["position_tier"] == "page_1")]
print(f"Weak-pick pattern: {len(weak):,} of {len(queue):,} flagged pages sit at the BOTTOM of page 1")
print("(avg_position > 7). Their low CTR is partly expected for their within-tier rank, so a single")
print("tier median slightly over-flags them. A stronger model would adjust for exact position, not")
print("just the tier. This is the baseline's known ceiling -- exactly what Week 5 must beat.\n")

# Leakage check: no label-derived or future-window columns in the rule's inputs.
used = ["impressions_90d", "avg_position", "ctr", "position_tier", "tier_median_ctr", "gap"]
banned = ["trend_direction", "trend_pct", "is_declining_label", "impressions_last_30d", "impressions_prev_30d"]
leaks = [c for c in used if c in banned]
print("Rule inputs:", used)
print("Banned (label/future) columns used:", leaks if leaks else "none")
assert not leaks, "leakage: a banned column is in the rule inputs"
print("Leakage check passed: the baseline uses only current-window observable signals.")

Weak-pick pattern: 888 of 2,920 flagged pages sit at the BOTTOM of page 1
(avg_position > 7). Their low CTR is partly expected for their within-tier rank, so a single
tier median slightly over-flags them. A stronger model would adjust for exact position, not
just the tier. This is the baseline's known ceiling -- exactly what Week 5 must beat.

Rule inputs: ['impressions_90d', 'avg_position', 'ctr', 'position_tier', 'tier_median_ctr', 'gap']
Banned (label/future) columns used: none
Leakage check passed: the baseline uses only current-window observable signals.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.